In [1]:
import os

In [2]:
%pwd

'c:\\Users\\H_KhaN\\OneDrive\\Desktop\\Mlops_project_wine_quality\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\H_KhaN\\OneDrive\\Desktop\\Mlops_project_wine_quality'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir : Path
    data_path : Path



    

In [6]:
from src.Mlops_project_quality_predictor.constants import *
from src.Mlops_project_quality_predictor.utils.common import read_yaml, create_directories

In [13]:
class ConfigurationManager:
    """
    Central place to load project configuration and build typed config
    objects (e.g., DataTransformationConfig) for each pipeline stage.
    """

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,   # Path to config.yaml (paths, directories, artifacts)
        params_filepath=PARAMS_FILE_PATH,   # Path to params.yaml (model/hyperparameters)
        schema_filepath=SCHEMA_FILE_PATH):  # Path to schema.yaml (expected columns & dtypes)

        # Read each YAML file and store it as an attribute.
        # read_yaml returns a ConfigBox so keys can be accessed
        # with dot notation (e.g., self.config.data_transformation)
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        # Make sure the top-level artifacts folder exists before any stage runs
        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        """Build and return the config for the data transformation stage."""

        # Pull the data_transformation section out of config.yaml
        config = self.config.data_transformation

        # Create the folder where transformation outputs will be saved
        create_directories([config.root_dir])

        # Wrap the values in a dataclass so the next stage gets a typed object
        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path
        )

        # Return it (without this line the method returns None)
        return data_transformation_config

In [14]:
import os
from src.Mlops_project_quality_predictor.utils import logger
from  sklearn.model_selection import train_test_split
import pandas as pd



In [15]:
class DataTransformation:
    """
    Handles the data transformation stage of the pipeline.
    Currently it only splits the dataset into train and test sets.
    """

    def __init__(self, config: DataTransformationConfig):
        # Store the config object (root_dir, data_path) built by ConfigurationManager
        self.config = config

    def train_test_splitting(self):
        """
        Read the raw dataset, split it into train/test sets,
        and save both as CSV files in the transformation folder.
        """

        # Load the input data from the path given in the config
        data = pd.read_csv(self.config.data_path)

        # Split into train and test sets
        # (sklearn default: 75% train / 25% test, shuffled, no fixed seed)
        train, test = train_test_split(data)

        # Save the training set; index=False avoids writing the row index as an extra column
        train.to_csv(os.path.join(self.config.root_dir, 'train.csv'), index=False)

        # Save the test set in the same folder
        test.to_csv(os.path.join(self.config.root_dir, 'test.csv'), index=False)

        # Log that the split completed and record the resulting shapes
        logger.info("Split data into training and test sets")
        logger.info(train.shape)
        logger.info(test.shape)

        # Also print the shapes to the console for quick feedback
        print(train.shape)
        print(test.shape)


In [16]:
try:
    config_manager = ConfigurationManager()
    data_transformation_config = config_manager.get_data_transformation_config()

    data_transformation = DataTransformation(config=data_transformation_config)  # instance
    data_transformation.train_test_splitting()

except Exception as e:
    raise e

[2026-10-15 00:01:40,092:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\config\config.yaml]
[2026-10-15 00:01:40,097:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\params.yaml]
[2026-10-15 00:01:40,104:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\schema.yaml]
[2026-10-15 00:01:40,106:INFO:common:created directory at: artifacts]
[2026-10-15 00:01:40,108:INFO:common:created directory at: artifacts/data_transformation]
[2026-10-15 00:01:40,186:INFO:2770374903:Split data into training and test sets]
[2026-10-15 00:01:40,188:INFO:2770374903:(1199, 12)]
[2026-10-15 00:01:40,189:INFO:2770374903:(400, 12)]
(1199, 12)
(400, 12)
